### 02 - building the user feature table

In [1]:
from pathlib import Path
import duckdb

ROOT = Path.cwd().parent
OUT  = ROOT / "data"

con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")

def pq(name):
    """Return the quoted path of a Parquet file in data/, ready to use in SQL."""
    return f"'{(OUT / (name + '.parquet')).as_posix()}'"

WINDOW = "ts >= '2010-08-01' AND ts < '2010-10-01'"

print(pq("entries"))


'c:/Users/HP/Downloads/Data Mining I/Project/data/entries.parquet'


### Restricting the acitivity of the study window

In [2]:
con.sql(f"CREATE OR REPLACE VIEW posts    AS SELECT * FROM {pq('entries')}  WHERE {WINDOW}")
con.sql(f"CREATE OR REPLACE VIEW comments AS SELECT * FROM {pq('comments')} WHERE {WINDOW}")
con.sql(f"CREATE OR REPLACE VIEW likes    AS SELECT * FROM {pq('likes')}    WHERE {WINDOW}")

# who wrote each post (ALL posts, also older ones, so we can find the author of any commented post)
con.sql(f"CREATE OR REPLACE VIEW post_authors AS SELECT post_id, user_id AS author FROM {pq('entries')}")

con.sql("""
    SELECT (SELECT count(*) FROM posts)    AS posts_in_window,
           (SELECT count(*) FROM comments) AS comments_in_window,
           (SELECT count(*) FROM likes)    AS likes_in_window
""")


┌─────────────────┬────────────────────┬─────────────────┐
│ posts_in_window │ comments_in_window │ likes_in_window │
│      int64      │       int64        │      int64      │
├─────────────────┼────────────────────┼─────────────────┤
│        12437376 │            3618896 │          540332 │
└─────────────────┴────────────────────┴─────────────────┘

### Choosing the population: who is active, and who goes into the analysis?

In [3]:
con.sql(f"""
    WITH activity AS (
        SELECT user_id,
               max(posted)    AS posted,
               max(commented) AS commented,
               max(liked)     AS liked
        FROM (
            SELECT DISTINCT user_id, 1 AS posted, 0 AS commented, 0 AS liked FROM posts
            UNION ALL
            SELECT DISTINCT user_id, 0, 1, 0 FROM comments
            UNION ALL
            SELECT DISTINCT user_id, 0, 0, 1 FROM likes
        )
        GROUP BY user_id
    )
    SELECT coalesce(u.type, 'not in users.csv') AS account_type,
           a.posted, a.commented, a.liked,
           count(*) AS n_accounts
    FROM activity AS a
    LEFT JOIN {pq('users')} AS u ON a.user_id = u.user_id
    GROUP BY ALL
    ORDER BY account_type DESC, n_accounts DESC
""")


┌──────────────────┬────────┬───────────┬───────┬────────────┐
│   account_type   │ posted │ commented │ liked │ n_accounts │
│     varchar      │ int32  │   int32   │ int32 │   int64    │
├──────────────────┼────────┼───────────┼───────┼────────────┤
│ user             │      1 │         0 │     0 │     163218 │
│ user             │      1 │         1 │     0 │      27539 │
│ user             │      1 │         1 │     1 │       9391 │
│ user             │      0 │         1 │     0 │       1476 │
│ user             │      1 │         0 │     1 │       1187 │
│ user             │      0 │         0 │     1 │        519 │
│ user             │      0 │         1 │     1 │        494 │
│ not in users.csv │      1 │         0 │     0 │      36621 │
│ not in users.csv │      1 │         1 │     0 │       5885 │
│ not in users.csv │      0 │         1 │     0 │       2161 │
│ not in users.csv │      1 │         1 │     1 │       1819 │
│ not in users.csv │      0 │         1 │     1 │      

### Choosing the population (who is in the analysis?)

**Goal:** decide which accounts become rows in the user feature table (and later points in the clustering).

**What we did:**
1. For every account active in the study window (1 Aug – 30 Sep 2010), recorded **whether** it posted, commented and/or liked (`UNION ALL` of three lists  `GROUP BY user_id` with `max`).
2. Added the account type from `users.parquet` (user / group / not in users.csv).
3. Counted the accounts per combination of activities, to see how activity overlaps before deciding.

**Result:**

- **203,824 individual users** were active in the window; **201,335 (98.8%) posted** at least once.
- Only **2,489 users (1.2%)** commented or liked without ever posting.
- **163,218 users (81% of posters) only post** - they never comment or like - first sign of a "broadcaster" behaviour.

**Decision: population = individual users (type `user`) with at least one post in the window - 201,335 users.**

**Why:**
- Several features describe *how* a user posts (share of native posts, share of images/videos). They cannot be calculated for users with zero posts.
- Requiring a post loses only 1.2% of active users.
- **Groups (9,930)** are excluded: they are communities, not individual people.
- **Accounts missing from users.csv (48,695)** are excluded: their type is unknown (they could be groups); probably deleted before the October snapshot - *limitation*.
- Users with no activity at all leave no data and cannot be clustered - in this population, a "passive" user means a **low-activity poster**.

The list of selected users is saved as the table `population`, the starting point for all features.


### Creating the population table

In [4]:
con.sql(f"""
    CREATE OR REPLACE TABLE population AS
    SELECT DISTINCT p.user_id
    FROM posts AS p
    JOIN {pq('users')} AS u ON p.user_id = u.user_id
    WHERE u.type = 'user'
""")
con.sql("SELECT count(*) AS n_users FROM population")


┌─────────┐
│ n_users │
│  int64  │
├─────────┤
│  201335 │
└─────────┘

### Posting features

In [5]:
con.sql("""
    CREATE OR REPLACE TABLE f_posts AS
    SELECT user_id,
           count(*)                        AS n_posts,
           count(DISTINCT source_name)     AS n_sources,
           avg((source_name = '')::INT)    AS pct_native,
           avg((n_img > 0)::INT)           AS pct_image,
           avg((n_vid > 0)::INT)           AS pct_video
    FROM posts
    GROUP BY user_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Posting features (`f_posts`)

**Goal:** describe *how much* and *how* each user posts. One row per author, from the posts in the window.

| Feature | Definition | Attribute type | What it captures |
|---|---|---|---|
| `n_posts` | number of posts | ratio (count) | publishing volume — bots post extremely often |
| `n_sources` | number of distinct posting channels (Twitter, Ping.fm, native, …) | ratio (count) | aggregators pull content from many sources |
| `pct_native` | share of posts written directly on FriendFeed (empty `source_name`) | ratio, 0–1 | **low = automated cross-posting**, high = typed by a person |
| `pct_image` | share of posts with ≥ 1 image | ratio, 0–1 | content style |
| `pct_video` | share of posts with ≥ 1 video | ratio, 0–1 | content style |

*Trick used:* `avg((condition)::INT)` = share of rows where the condition is true (average of 0/1 values).

Shares (`pct_*`) are comparable between heavy and light users; raw counts are not.


### Comment features and a surprise

In [6]:
con.sql("""
    SELECT c.user_id = a.author                   AS on_own_post,
           count(*)                               AS n_comments,
           count(DISTINCT c.user_id)              AS n_commenters,
           round(100 * avg((c.source_name = '')::INT), 1) AS pct_native
    FROM comments AS c
    JOIN post_authors AS a ON c.post_id = a.post_id
    GROUP BY on_own_post
""")


┌─────────────┬────────────┬──────────────┬────────────┐
│ on_own_post │ n_comments │ n_commenters │ pct_native │
│   boolean   │   int64    │    int64     │   double   │
├─────────────┼────────────┼──────────────┼────────────┤
│ false       │    1809565 │        20215 │       96.9 │
│ true        │    1809214 │        41874 │       81.2 │
└─────────────┴────────────┴──────────────┴────────────┘

In [7]:
con.sql("""
    CREATE OR REPLACE TABLE f_comments_written AS
    SELECT c.user_id,
           count(*) FILTER (WHERE c.user_id <> a.author) AS n_comments_on_others,
           count(*) FILTER (WHERE c.user_id =  a.author) AS n_comments_on_own
    FROM comments AS c
    JOIN post_authors AS a ON c.post_id = a.post_id
    GROUP BY c.user_id
""")

con.sql("""
    CREATE OR REPLACE TABLE f_comments_received AS
    SELECT a.author AS user_id,
           count(*) AS n_comments_received
    FROM comments AS c
    JOIN post_authors AS a ON c.post_id = a.post_id
    WHERE c.user_id <> a.author
    GROUP BY a.author
""")


### Comment features (`f_comments_written`, `f_comments_received`)

**Goal:** measure conversation - commenting on others and receiving comments from others.

**First check - who comments on whose posts?** (comments in the window joined to the post's author)

| Comment is on… | Comments | Commenters | % native |
|---|---|---|---|
| own post | 1,809,214 | 41,874 | 81.2% |
| someone else's post | 1,809,565 | 20,215 | 96.9% |

**Finding: half of all comments are on the commenter's own post** (e.g. a note added to one's own shared link). That is not conversation with others, so counting all comments as "conversation" would distort the conversationalist archetype.

**Features created:**
- `n_comments_on_others` - comments written on other people's posts (**conversation**)
- `n_comments_on_own` - comments written on one's own posts
- `n_comments_received` - comments **by others** on the user's posts (grouped by the post's author; self-comments excluded)

Comments whose post is not in the data (orphans) are dropped by the `JOIN`, because their author is unknown.


### Like Features

In [8]:
con.sql("""
    CREATE OR REPLACE TABLE f_likes_given AS
    SELECT user_id, count(*) AS n_likes_given
    FROM likes
    GROUP BY user_id
""")

con.sql("""
    CREATE OR REPLACE TABLE f_likes_received AS
    SELECT a.author AS user_id, count(*) AS n_likes_received
    FROM likes AS l
    JOIN post_authors AS a ON l.post_id = a.post_id
    WHERE l.user_id <> a.author
    GROUP BY a.author
""")


### Like features (`f_likes_given`, `f_likes_received`)

**Goal:** measure lightweight interaction.

- `n_likes_given` - likes the user gave in the window (grouped by the liker)
- `n_likes_received` - likes **by others** on the user's posts (joined to the post's author, grouped by author)

Self-likes are excluded for consistency with comments (in practice there are none - FriendFeed did not allow liking one's own post).
Likes on posts not in the data (442 orphans in the window) are dropped from "received", since their author is unknown.


### Network and Services

In [9]:
con.sql(f"""
    CREATE OR REPLACE TABLE f_followers AS
    SELECT followed AS user_id, count(*) AS n_followers
    FROM {pq('follows')} GROUP BY followed
""")

con.sql(f"""
    CREATE OR REPLACE TABLE f_following AS
    SELECT follower AS user_id, count(*) AS n_following
    FROM {pq('follows')} GROUP BY follower
""")

con.sql(f"""
    CREATE OR REPLACE TABLE f_services AS
    SELECT user_id, count(*) AS n_services
    FROM {pq('services')} GROUP BY user_id
""")


### Network and service features (`f_followers`, `f_following`, `f_services`)

**Goal:** describe the user's position in the network and how many external services they connect.

| Feature | Source | Meaning |
|---|---|---|
| `n_followers` (in-degree) | `follows.parquet`, count per *followed* | how many accounts follow the user - **influence** |
| `n_following` (out-degree) | `follows.parquet`, count per *follower* | how many accounts the user follows - extreme values suggest **follow-spam bots** |
| `n_services` | `services.parquet`, count per user | number of linked external services - **cross-platform integration** (link to Question 2) |

These come from the **October snapshots**, so they are not restricted to the study window (limitation: measured ~2 weeks after the activity period).
Only simple counts are used - no further graph analysis (scope agreed with the teacher).


### Assembling one row per user

In [10]:
con.sql(f"""
    CREATE OR REPLACE TABLE user_features AS
    SELECT p.user_id,
           fp.n_posts, fp.n_sources, fp.pct_native, fp.pct_image, fp.pct_video,
           coalesce(cw.n_comments_on_others, 0) AS n_comments_on_others,
           coalesce(cw.n_comments_on_own,    0) AS n_comments_on_own,
           coalesce(cr.n_comments_received,  0) AS n_comments_received,
           coalesce(lg.n_likes_given,        0) AS n_likes_given,
           coalesce(lr.n_likes_received,     0) AS n_likes_received,
           coalesce(fo.n_followers,          0) AS n_followers,
           coalesce(fg.n_following,          0) AS n_following,
           coalesce(sv.n_services,           0) AS n_services,
           u.has_description
    FROM population                AS p
    JOIN      f_posts              AS fp ON p.user_id = fp.user_id
    LEFT JOIN f_comments_written   AS cw ON p.user_id = cw.user_id
    LEFT JOIN f_comments_received  AS cr ON p.user_id = cr.user_id
    LEFT JOIN f_likes_given        AS lg ON p.user_id = lg.user_id
    LEFT JOIN f_likes_received     AS lr ON p.user_id = lr.user_id
    LEFT JOIN f_followers          AS fo ON p.user_id = fo.user_id
    LEFT JOIN f_following          AS fg ON p.user_id = fg.user_id
    LEFT JOIN f_services           AS sv ON p.user_id = sv.user_id
    JOIN {pq('users')}             AS u  ON p.user_id = u.user_id
""")

con.sql("SELECT count(*) AS n_rows, count(DISTINCT user_id) AS n_users FROM user_features")


┌────────┬─────────┐
│ n_rows │ n_users │
│ int64  │  int64  │
├────────┼─────────┤
│ 201335 │  201335 │
└────────┴─────────┘

### Assembling one row per user (`user_features`)

**Goal:** combine all feature tables into a single table with **one row per user in the population**.

**How:**
- Start from `population` (201,335 users) and attach each feature table by `user_id`.
- `JOIN f_posts`: every population member posted, so there is always a match.
- `LEFT JOIN` for all other tables: most users have no comments/likes and therefore no row there; a normal JOIN would drop them, a LEFT JOIN keeps them with NULL.
- `coalesce(x, 0)`: turns those NULLs into **0**, because "no row in the likes table" really means "zero likes" - a missing value whose meaning we know.
- `has_description` is taken from `users.parquet`.

**Check:** `n_rows = n_users = 201,335` - every user appears exactly once.
(This check caught a real problem: `users.csv` contained duplicate user IDs, which made some users appear 2–3 times after the join. Fixed in notebook 01 by deduplicating users.)

**Result:** 201,335 users × 15 features.


### Summary

In [11]:
con.sql("SUMMARIZE user_features").df()[["column_name", "min", "max", "avg", "q25", "q50", "q75"]]


,column_name,min,max,avg,q25,q50,q75
0,user_id,000,zzzzhe,None,None,None,None
1,n_posts,1,166965,49.256329003898976,3,10,31
2,n_sources,1,2753,2.266903419673678,1,1,2
3,pct_native,0.0,1.0,0.08718071130760235,0.0,0.0,0.0
4,pct_image,0.0,1.0,0.11884969668066574,0.0,0.0,0.08482776880542292
5,pct_video,0.0,1.0,0.04386687093476755,0.0,0.0,0.0
6,n_comments_on_others,0,8757,6.081257605483398,0,0,0
7,n_comments_on_own,0,12197,6.811696923038717,0,0,0
8,n_comments_received,0,16931,7.3142771996920555,0,0,0
9,n_likes_given,0,3153,1.7212456850522762,0,0,0


In [13]:
con.sql(f"COPY user_features TO {pq('user_features')} (FORMAT parquet)")
print("saved")

saved


### Summary statistics and saving

**Goal:** a first look at the distribution of every feature, then save the table.

| Feature | Median | 75th pct | Max | Mean |
|---|---|---|---|---|
| n_posts | 10 | 31 | 166,965 | 49.3 |
| n_sources | 1 | 2 | 2,753 | 2.3 |
| pct_native | 0 | 0 | 1 | 0.09 |
| n_comments_on_others | 0 | 0 | 8,757 | 6.1 |
| n_comments_received | 0 | 0 | 16,931 | 7.3 |
| n_likes_given | 0 | 0 | 3,153 | 1.7 |
| n_followers | 19 | 74 | 43,733 | 93.4 |
| n_following | 10 | 41 | 105,618 | 90.2 |
| n_services | 3 | 6 | 55 | 4.3 |

**What it shows:**
1. **Extreme skew:** for every count, the mean is far above the median. One account posted 166,965 times in two months (~2,700 posts per day) - certainly automated.
2. **Mostly zeros for interaction:** ~94% of users never comment on others, receive no comments, and give no likes.
3. **Most posting is automated:** median `pct_native` = 0 - the typical user publishes only through external tools.

**Consequence for the next step:** raw counts cannot go into distance-based clustering as they are - Euclidean distance would be dominated by a few extreme accounts. - histograms/box plots, then **log transform + scaling** (Step 3).

Saved as `data/user_features.parquet`.
